# ICA fit

Fit ICA on the epochs kept in 02, label the components with ICLabel, select
the components to remove and save the selection in the decision file. The data
is not changed here; 04 applies the selection.

An optional manual selection (last section) is saved in the same decision
file, next to the automatic one. 04 uses the manual selection when there is one.

In [ ]:
subject_id = "101"
# set by the runner (papermill); None means: find settings.toml from cwd
config_path = None
# set by the runner, e.g. a test folder for derivatives_root
path_overrides = None
# True: show the component review widget at the end (interactive use only)
review = False

In [ ]:
from functools import partial

import mne
from IPython.display import display
from mne_icalabel import label_components

from spectral.helpers import load_or_create_report, save_subject_report
from spectral.ica import (compute_ica_limited, ica_overview,
                          plot_component_properties, review_components,
                          write_ica_decision)
from spectral.utils import ProjectPaths, print_timestamp

paths = ProjectPaths(subject_id, config_path=config_path, path_overrides=path_overrides)
subject_id = paths.subject_id  # zero-padded, as in the file names from 01
paths.create_directories()
print_timestamp("Setting up project paths")
paths.show()

# One report per subject: 01 creates it, each step adds its own section.
REPORT_SECTION = "03 ICA fit"
report = load_or_create_report(paths, subject_id)
add_fig = partial(report.add_figure, section=REPORT_SECTION, replace=True)

mne.viz.set_browser_backend("matplotlib")
mne.set_config("MNE_BROWSER_THEME", "light")

## Data

In [ ]:
clean_raw = mne.io.read_raw_fif(
    paths.preprocessed / f"sub-{subject_id}_annotated_filtered_raw.fif", preload=True
)
# 01 already removed 3 s at each end and this removes 3 s more. Kept for now:
# the ECG correlation below uses clean_raw, so changing it changes results
# (open question in CHANGELOG.md).
total_duration = clean_raw.times[-1]
clean_raw = clean_raw.crop(tmin=3.0, tmax=total_duration - 3, include_tmax=True)

# Epochs kept in 02; the ICA is fitted on these.
epochs_good = mne.read_epochs(paths.epochs / f"sub-{subject_id}_good_epochs-epo.fif")

report.add_raw(raw=clean_raw, title=f"{REPORT_SECTION}: raw data", psd=True,
               butterfly=True, scalings="auto", replace=True)
if clean_raw.info["bads"]:
    # Plotting bads can raise if a channel was marked bad and then dropped.
    try:
        bad_channel_plot = clean_raw.plot(
            picks=clean_raw.info["bads"],
            duration=min(300.0, clean_raw.times[-1]),
            scalings=dict(eeg=1e-4),
            show_scrollbars=False,
            show=False,
        )
        add_fig(bad_channel_plot, title="Bad channels (interpolated in 04)",
                tags=("bad_channels",))
    except Exception as e:
        print(f"Could not plot bad channels: {e}")
else:
    print("No bad channels marked.")

## Fit

In [ ]:
# ICA parameters come from settings.toml ([cleaning]): a fixed number of
# components (variance_threshold=None). The method (picard, extended) is
# fixed inside compute_ica_limited.
_cl = paths.config["cleaning"]
ica = compute_ica_limited(epochs_good,
                          variance_threshold=None,
                          max_cap=_cl["ica_n_components"],
                          random_state=_cl["ica_random_state"])
ica.save(paths.analysis / f"sub-{subject_id}_my_ica_model-ica.fif", overwrite=True)

explained = ica.get_explained_variance_ratio(epochs_good)

# Data per component: unique samples (overlapping parts of epochs counted once)
# per squared number of components. At least 30 is the usual rule (HAPPE,
# Gabard-Durnam et al. 2018, after Onton & Makeig).
import numpy as np
_onsets = np.sort(epochs_good.events[:, 0])
_n = len(epochs_good.times)
_unique = int(np.minimum(np.diff(_onsets), _n).sum() + _n)
_k = _unique / ica.n_components_ ** 2
print(f"{_unique} unique samples, {_k:.1f} per squared component (30 or more recommended)")
if _k < 30:
    report.add_html(f"<p><b>Little data for the ICA:</b> {_unique} unique samples, "
                    f"{_k:.1f} per squared component (30 or more recommended). "
                    f"Check this subject's decomposition.</p>",
                    title="ICA: little data per component", section=REPORT_SECTION, replace=True)
print(f"{ica.n_components_} components, explaining "
      + ", ".join(f"{ratio:.1%} of {ch_type}" for ch_type, ratio in explained.items()))

## Classify

In [ ]:
# Components correlated with the ECG channel. Skipped if there is no ECG
# channel (find_bads_ecg raises an error then).
if mne.pick_types(clean_raw.info, meg=False, ecg=True).size:
    ecg_indices, ecg_scores = ica.find_bads_ecg(
        clean_raw, method="correlation", threshold="auto"
    )
else:
    ecg_indices, ecg_scores = [], None
    print("No ECG channel, find_bads_ecg skipped.")

# ICLabel on epochs_good, the data the ICA was fitted on (clean_raw also
# contains segments that were not used for the fit).
# 01 filters 1-100 Hz, the band ICLabel was trained on.
ic_labels = label_components(epochs_good, ica, method="iclabel")

## Decide

In [ ]:
# A component is removed if ICLabel classifies it as an artifact with a
# probability above the class threshold ([cleaning] ica_label_thresholds), or
# if it correlates with the ECG and ICLabel does not classify it as brain
# (ecg_reject_labels).
_thresholds = _cl["ica_label_thresholds"]
_ecg_reject_labels = _cl["ecg_reject_labels"]
_labels, _probs = ic_labels["labels"], ic_labels["y_pred_proba"]

bad_indices = [
    i for i, label in enumerate(_labels)
    if label in _thresholds and _probs[i] > _thresholds[label]
]
ecg_kept = [i for i in ecg_indices if _labels[i] not in _ecg_reject_labels]
ecg_dropped = [i for i in ecg_indices if _labels[i] in _ecg_reject_labels]
if ecg_dropped:
    print("ECG hits ignored (ICLabel calls them brain): "
          + ", ".join(f"IC{i} p={_probs[i]:.2f}" for i in ecg_dropped))

final_picks = sorted(set(bad_indices + ecg_kept))
ica.exclude = final_picks  # only used for the plots below; 04 reads the decision file

# Overview of all components: ICLabel class, probability and share of
# variance (var_pct). Removed components are shown in red.
overview, overview_fig = ica_overview(
    ica, epochs_good, ic_labels, exclude=final_picks, thresholds=_thresholds
)
display(overview)
add_fig(overview_fig, title="ICA decision overview", tags=("ica",))
print(f"excluding {len(final_picks)} of {ica.n_components_} components, "
      f"{overview.loc[overview.excluded, 'var_pct'].sum():.1f}% of variance")

In [ ]:
# Removed components: properties (topography, spectrum, epoch image) and the
# first 20 s of their time course.
if final_picks:
    plot_component_properties(
        ica, clean_raw, picks=final_picks, ic_labels=ic_labels,
        report=report, section=REPORT_SECTION,
        save_path=paths.figures, subject=subject_id,
    )
    fig_sources = ica.plot_sources(
        clean_raw, picks=final_picks, start=0, stop=20, show_scrollbars=False
    )
    add_fig(fig_sources, title="Removed components, time course", tags=("ica",))

## Save the decision

In [ ]:
# The decision file stores the automatic selection, the ICLabel class of each
# component and a fingerprint of the decomposition. The fingerprint is used to
# check a later manual selection against a refitted ICA. Manual fields are not
# written here.
decision_path = paths.analysis / f"sub-{subject_id}_ica-decision.json"
decision = write_ica_decision(
    decision_path, ica,
    auto_detected=final_picks,
    labels={str(i): [_labels[i], float(_probs[i])] for i in range(ica.n_components_)},
    extra={"subject_id": str(subject_id),
           "ica_params": paths.config["cleaning"],
           "ecg_indices": [int(i) for i in ecg_indices]},
)
print(f"decomposition fingerprint : {decision['ica_fingerprint']}")
print(f"auto-detected components  : {decision['auto_detected']}")
print(f"decision file             : {decision_path}")

In [ ]:
save_subject_report(report, paths, subject_id)

## Manual review (optional)

Go through the components, mark the ones to remove and press "Save decision".
In a papermill run there is no input, so the automatic selection is used.

Saving adds `manual_exclude` to the decision file and keeps the automatic list.
The topographies used for the review are saved with it. 04 uses the manual
list, and stops with an error if the ICA was refitted and the components cannot
be matched. Rerun 04 and 05 for the subject afterwards.

For larger figures, run `%matplotlib widget` first.

In [ ]:
# Manual review, interactive only: set review = True in the parameters cell.
# The pipeline runs without it; a widget cannot be used there, and in
# runs/v0.3.0-dev it left sub-105 hanging in this cell.
if review:
    widget, review_state = review_components(
        ica, epochs_good, ic_labels,
        exclude=final_picks,          # start from the automatic selection
        decision_path=decision_path,  # written above
        timeseries=clean_raw,         # continuous data instead of concatenated epochs
        window_s=20.0,
        psd_fmax=100.0,               # spectrum up to 100 Hz (01 filters 1-100 Hz)
    )
    display(widget)